# Representative Statements Workbench

The purpose of this notebook is to test the Agora representative statement selection and compare it to the Polis output for the same clusters. 

In [ ]:
from pathlib import Path
from pprint import pprint
from types import SimpleNamespace
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", None)

REPO_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "pyproject.toml").exists() and (parent / "reddwarf").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import reddwarf
imported_repo_root = Path(reddwarf.__file__).resolve().parents[1]
if imported_repo_root != REPO_ROOT:
    raise RuntimeError(
        f"Notebook imported reddwarf from {imported_repo_root}, expected {REPO_ROOT}. Restart the kernel and rerun from the repo clone."
    )

from reddwarf.data_loader import Loader
from reddwarf.utils.matrix import generate_raw_matrix
from reddwarf.utils.polismath import extract_data_from_polismath
from reddwarf.utils.stats import (
    calculate_comment_statistics_dataframes,
    rank_representative_statements,
    select_representative_statements,
)
from reddwarf.utils.statements import process_statements


## Settings

- `DATA_SOURCE = "fixture"` uses local test fixtures
- `DATA_SOURCE = "polis_id"` fetches a live report by Polis report id
- This notebook always reuses platform `math_data["group-clusters"]` for both Polis and Agora representative selection so the cluster source stays fixed.


In [ ]:
DATA_SOURCE = "fixture"  # fixture | polis_id
FIXTURE_DIR = "../../tests/fixtures/below-100-ptpts"
POLIS_ID = ""

RANDOM_STATE = 42
FDR_RATE = 0.10
DIVISIVE_N_RESAMPLES = 399
DIVISIVE_RANDOM_STATE = 42
STRONG_EFFECT_MIN = 1.0
STRONG_SMALL_GROUP_CUTOFF = 5
STRONG_LARGE_GROUP_PARTICIPATION_MIN = 0.8
STRONG_P_MAX = 0.05
TOP_N = 10


In [ ]:
if DATA_SOURCE == "fixture":
    loader = Loader(filepaths=[
        f"{FIXTURE_DIR}/votes.json",
        f"{FIXTURE_DIR}/comments.json",
        f"{FIXTURE_DIR}/conversation.json",
        f"{FIXTURE_DIR}/math-pca2.json",
    ])
    dataset_label = FIXTURE_DIR
elif DATA_SOURCE == "polis_id":
    loader = Loader(polis_id=POLIS_ID)
    dataset_label = POLIS_ID
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

_, _, mod_out_statement_ids, meta_statement_ids = process_statements(loader.comments_data)

stmt_text = {}
for comment in loader.comments_data:
    sid = comment.get("tid", comment.get("statement_id"))
    stmt_text[int(sid)] = comment.get("txt", "")

if not getattr(loader, "math_data", None) or "group-clusters" not in loader.math_data:
    raise ValueError("This notebook requires platform math_data with group-clusters.")

print(f"Dataset: {dataset_label}")
print(f"Votes: {len(loader.votes_data)}")
print(f"Statements: {len(loader.comments_data)} total")
print(f"Moderated out / meta: {len(mod_out_statement_ids)}")
print("Cluster source: platform_group_clusters")


## Building Shared Context for Both Pipelines

This notebook reuses the platform (from Polis) participant-to-group assignments from `math_data["group-clusters"]` and does not rerun clustering (for the sake of comparing the pipelines on identical clusters)


In [ ]:
def get_platform_cluster_context(loader, valid_participant_ids):
    platform_ids, platform_labels = extract_data_from_polismath(loader.math_data)
    valid_ids = set(valid_participant_ids)
    platform_pairs = [
        (int(pid), int(label))
        for pid, label in zip(platform_ids, platform_labels)
        if pid in valid_ids
    ]
    participant_ids_to_cluster = [pid for pid, _ in platform_pairs]
    cluster_labels = np.asarray([label for _, label in platform_pairs], dtype=int)
    return participant_ids_to_cluster, cluster_labels

raw_vote_matrix = generate_raw_matrix(votes=loader.votes_data)
participant_ids_to_cluster, cluster_labels = get_platform_cluster_context(loader, raw_vote_matrix.index)
clustered_vote_matrix = raw_vote_matrix.loc[participant_ids_to_cluster, :]

grouped_stats_df, _ = calculate_comment_statistics_dataframes(
    vote_matrix=clustered_vote_matrix,
    cluster_labels=cluster_labels,
    consensus_mode="standard",
)

ranked_repness = rank_representative_statements(
    grouped_stats_df=grouped_stats_df,
    vote_matrix=clustered_vote_matrix,
    cluster_labels=cluster_labels,
    mod_out_statement_ids=mod_out_statement_ids,
    fdr_rate=FDR_RATE,
    divisive_n_resamples=DIVISIVE_N_RESAMPLES,
    divisive_random_state=DIVISIVE_RANDOM_STATE,
    strong_effect_min=STRONG_EFFECT_MIN,
    strong_small_group_cutoff=STRONG_SMALL_GROUP_CUTOFF,
    strong_large_group_participation_min=STRONG_LARGE_GROUP_PARTICIPATION_MIN,
    strong_p_max=STRONG_P_MAX,
)

participants_df = pd.DataFrame(index=raw_vote_matrix.index)
participants_df["to_cluster"] = participants_df.index.isin(participant_ids_to_cluster)
participants_df["cluster_id"] = pd.Series(cluster_labels, index=participant_ids_to_cluster, dtype="Int64")

agora_result = SimpleNamespace(
    raw_vote_matrix=raw_vote_matrix,
    group_comment_stats=grouped_stats_df,
    ranked_repness=ranked_repness,
    participants_df=participants_df,
)

print(f"Agora groups: {sorted(agora_result.ranked_repness.keys())}")
print(f"Agora clustered participants: {int(agora_result.participants_df['to_cluster'].sum())}")


In [ ]:
polis_repness_same_groups = select_representative_statements(
    grouped_stats_df=agora_result.group_comment_stats,
    mod_out_statement_ids=mod_out_statement_ids,
    pick_max=5,
    confidence=0.9,
)

print(f"Polis groups on same cluster source: {sorted(polis_repness_same_groups, key=int)}")


## View Representative Statements for Both Pipelines

In [ ]:
def truncate(text, max_len=100):
    text = text or ""
    return text if len(text) <= max_len else text[:max_len] + "..."

def vote_breakdown(na, nd, ns):
    na = int(na)
    nd = int(nd)
    ns = int(ns)
    npass = max(ns - na - nd, 0)
    votes = f"{na}/{nd}/{npass}/{ns}"
    if ns == 0:
        pct = "0.0%/0.0%/0.0%"
    else:
        pct = f"{100*na/ns:.1f}%/{100*nd/ns:.1f}%/{100*npass/ns:.1f}%"
    return votes, pct

def polis_group_df(group_id):
    rows = []
    for rank, row in enumerate(polis_repness_same_groups.get(group_id, []), start=1):
        statement_id = int(row["tid"])
        grouped_row = agora_result.group_comment_stats.loc[(group_id, statement_id)]
        in_votes, in_pct = vote_breakdown(grouped_row["na"], grouped_row["nd"], grouped_row["ns"])
        rows.append({
            "rank": rank,
            "statement_id": statement_id,
            "repful_for": row.get("repful-for"),
            "best_agree": bool(row.get("best-agree", False)),
            "n_success": int(row["n-success"]),
            "n_trials": int(row["n-trials"]),
            "p_success": float(row["p-success"]),
            "p_test": float(row["p-test"]),
            "repness": float(row["repness"]) if row.get("repness") is not None else None,
            "repness_test": float(row["repness-test"]) if row.get("repness-test") is not None else None,
            "In Votes": in_votes,
            "In %": in_pct,
            "text": truncate(stmt_text.get(statement_id, "?")),
        })
    return pd.DataFrame(rows)

def agora_group_df_detailed(group_id, top_n=None):
    rows = []
    statements = agora_result.ranked_repness[group_id]
    if top_n is not None:
        statements = statements[:top_n]
    for statement in statements:
        npass = int(statement.ns - statement.na - statement.nd)
        npass_out = int(statement.ns_out - statement.na_out - statement.nd_out)
        in_votes, in_pct = vote_breakdown(statement.na, statement.nd, statement.ns)
        out_votes, out_pct = vote_breakdown(statement.na_out, statement.nd_out, statement.ns_out)
        rows.append({
            "rank": int(statement.rank),
            "st_id": int(statement.statement_id),
            "repful_for": statement.repful_for,
            "selected": bool(statement.selected),
            "strength": statement.signal_strength,
            "effect_size": float(statement.effect_size),
            "p_value": float(statement.p_value),
            "adjusted_p_value": float(statement.adjusted_p_value),
            "agree_effect": float(statement.agree_effect),
            "disagree_effect": float(statement.disagree_effect),
            "divisive_effect": float(statement.divisive_effect),
            "na": int(statement.na),
            "nd": int(statement.nd),
            "npass": npass,
            "ns": int(statement.ns),
            "In Votes": in_votes,
            "In %": in_pct,
            "na_out": int(statement.na_out),
            "nd_out": int(statement.nd_out),
            "npass_out": npass_out,
            "ns_out": int(statement.ns_out),
            "Out Votes": out_votes,
            "Out %": out_pct,
            "divisiveness": float(statement.divisiveness),
            "text": truncate(stmt_text.get(statement.statement_id, "?")),
        })
    return pd.DataFrame(rows)

print(f"Dataset: {dataset_label}")
for gid in sorted(agora_result.ranked_repness):
    print(f"\n=== Group {gid} ===\n")
    print("Polis representative statements (same cluster source):")
    display(polis_group_df(gid))
    print("\nAgora representative statements:")
    display(agora_group_df_detailed(gid, top_n=TOP_N))


## Summary

Comparison of Polis top 5 ids, Agora top-ranked ids and Agora `selected=True` ids per group.


In [ ]:
summary_rows = []
for gid in sorted(agora_result.ranked_repness):
    polis_top_ids = [int(row["tid"]) for row in polis_repness_same_groups.get(gid, [])]
    agora_top_ids = [statement.statement_id for statement in agora_result.ranked_repness[gid][:TOP_N]]
    agora_selected_ids = [statement.statement_id for statement in agora_result.ranked_repness[gid] if statement.selected]
    summary_rows.append({
        "group_id": gid,
        "polis_top_ids": polis_top_ids,
        "agora_top_ids": agora_top_ids,
        "agora_selected_ids": agora_selected_ids,
        "n_agora_selected": len(agora_selected_ids),
    })

display(pd.DataFrame(summary_rows))
